# CELL 1 — Load preprocessed HDF5 data

In [33]:
import os
print(os.listdir("artifacts"))

[]


In [25]:
import h5py, os

H5_PATH = os.path.join("..", "artifacts", "wildlife_data.h5")  # go up one level from notebooks/

with h5py.File(H5_PATH, "r") as f:
    X_train = f["X_train"][:]
    y_train = f["y_train"][:]
    X_val   = f["X_val"][:]
    y_val   = f["y_val"][:]
    X_test  = f["X_test"][:]   # <-- this is what was missing
    y_test  = f["y_test"][:]   # <-- and this
    classes = f["classes"][:]

print("Loaded HDF5 data successfully")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print("Classes:", classes)



Loaded HDF5 data successfully
Train: (1052, 224, 224, 3), Val: (226, 224, 224, 3), Test: (226, 224, 224, 3)
Classes: [b'buffalo' b'elephant' b'rhino' b'zebra']


# CELL 2 — Prepare test set arrays

In [27]:
X_test_all = X_test
y_test_all = y_test.astype(int).ravel()   # flatten to 1D

print(f"Test set: {X_test_all.shape} | Labels: {np.bincount(y_test_all)}")



Test set: (226, 224, 224, 3) | Labels: [678 226]


# CELL 3 — Core SHAP helper functions

In [28]:
def build_background(X_train, n=100):
    """Collect n training images to use as the SHAP reference baseline."""
    idx = np.random.choice(len(X_train), n, replace=False)
    return X_train[idx]

def run_shap_cnn(model, background, images):
    explainer   = shap.GradientExplainer(model, background)
    shap_values = explainer.shap_values(images)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    return np.array(shap_values)

print("Helper functions defined.")


Helper functions defined.


# CELL 4 — SECTION A: Standard pixel heatmaps 

In [29]:
for model_name, model_path in MODELS.items():
    if not os.path.exists(model_path):
        print(f"[SKIP] {model_name}: {model_path} not found")
        continue

    print(f"\n[{model_name}] Computing heatmaps...")
    model      = tf.keras.models.load_model(model_path)
    shap_vals  = run_shap_cnn(model, background, sample_imgs)
    shap_vals  = shap_vals / np.max(np.abs(shap_vals))  # normalize

    for i, img in enumerate(sample_imgs):
        shap_map = np.mean(np.abs(shap_vals[i]), axis=-1)
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.imshow(shap_map, cmap="inferno", alpha=0.6)
        plt.axis("off")
        plt.title(f"{model_name} — Sample {i+1}")
        out = f"artifacts/shap_{model_name}_sample{i+1}.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"[{model_name}] Saved: {out}")


[SKIP] MobileNetV2: artifacts/MobileNetV2_best.h5 not found
[SKIP] ResNet50: artifacts/ResNet50_best.h5 not found
[SKIP] EfficientNetB0: artifacts/EfficientNetB0_best.h5 not found


# CELL 5 — SECTION B: Correct vs incorrect predictions

In [30]:
N_CORRECT   = 3   # examples to show from correctly classified
N_INCORRECT = 3   # examples to show from incorrectly classified

for model_name, model_path in MODELS.items():
    if not os.path.exists(model_path):
        print(f"[SKIP] {model_name}: {model_path} not found")
        continue

    print(f"\n[{model_name}] Correct vs incorrect analysis...")
    model  = tf.keras.models.load_model(model_path)
    raw_preds = model.predict(X_test_all, verbose=0)
    tf.keras.backend.clear_session()

    # Handle binary vs multi-class outputs
    if raw_preds.ndim == 2 and raw_preds.shape[1] == 1:
        preds = (raw_preds.ravel() > 0.5).astype(int)
    else:
        preds = np.argmax(raw_preds, axis=1)

    correct_idx   = np.where(preds == y_test_all)[0]
    incorrect_idx = np.where(preds != y_test_all)[0]

    print(f"  Correct   : {len(correct_idx)} / {len(y_test_all)}")
    print(f"  Incorrect : {len(incorrect_idx)} / {len(y_test_all)}")

    # Pick up to N examples from each group
    c_imgs = X_test_all[correct_idx[:N_CORRECT]]
    w_imgs = X_test_all[incorrect_idx[:N_INCORRECT]] if len(incorrect_idx) else np.array([])

    # Compute SHAP values
    model = tf.keras.models.load_model(model_path)
    c_shap = run_shap_cnn(model, background, c_imgs) if len(c_imgs) else None
    w_shap = run_shap_cnn(model, background, w_imgs) if len(w_imgs) else None
    tf.keras.backend.clear_session()

    # Plot correct predictions
    if c_shap is not None:
        c_shap = c_shap / np.max(np.abs(c_shap))  # normalize
        for i, img in enumerate(c_imgs):
            shap_map = np.mean(np.abs(c_shap[i]), axis=-1)
            plt.figure(figsize=(4,4))
            plt.imshow(img)
            plt.imshow(shap_map, cmap="inferno", alpha=0.6)
            plt.axis("off")
            plt.title(f"{model_name} — Correct (True {y_test_all[correct_idx[i]]})", color="green")
            out = f"artifacts/shap_{model_name}_correct{i+1}.png"
            plt.savefig(out, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"  [B] Saved: {out}")

    # Plot incorrect predictions
    if w_shap is not None:
        w_shap = w_shap / np.max(np.abs(w_shap))  # normalize
        for i, img in enumerate(w_imgs):
            shap_map = np.mean(np.abs(w_shap[i]), axis=-1)
            plt.figure(figsize=(4,4))
            plt.imshow(img)
            plt.imshow(shap_map, cmap="inferno", alpha=0.6)
            plt.axis("off")
            plt.title(
                f"{model_name} — Incorrect (Pred {preds[incorrect_idx[i]]}, True {y_test_all[incorrect_idx[i]]})",
                color="red"
            )
            out = f"artifacts/shap_{model_name}_incorrect{i+1}.png"
            plt.savefig(out, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"  [B] Saved: {out}")
    else:
        print(f"  [B] No incorrect predictions — skipping incorrect plot.")


[SKIP] MobileNetV2: artifacts/MobileNetV2_best.h5 not found
[SKIP] ResNet50: artifacts/ResNet50_best.h5 not found
[SKIP] EfficientNetB0: artifacts/EfficientNetB0_best.h5 not found


# CELL 6 — SECTION C: Mean absolute SHAP

In [ ]:
im = axes[0].imshow(mean_shap, cmap="inferno")
axes[1].imshow(avg_img)
axes[1].imshow(mean_shap, cmap="inferno", alpha=0.5)


# CELL 7 — SECTION D: Side‑by‑side model comparison

In [ ]:
im = ax.imshow(mean_shap, cmap="inferno", vmin=0, vmax=vmax)
